## Camada Bronze

In [0]:
import os

### Uso do schema

In [0]:
catalog = "mvp_data_engeneering"
schema = "bronze"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

###Cópia para DataFrames

In [0]:
schema_origem = "staging"
volume_origem = "brazilian_ecommerce"

path_to_volume = f"/Volumes/{catalog}/{schema_origem}/{volume_origem}/"

dfs = {}
for f in os.listdir(path_to_volume):
    full_path = os.path.join(path_to_volume, f)
    f = "df_" + f.replace(".csv","").replace("olist_","").replace("_dataset","")
    dfs[f] = (spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(full_path))
    print(f"DataFrame {f} criado com sucesso")

###Visualização dos DataFrames

In [0]:
for name, df in dfs.items():
    print(f"DataFrame {name}")
    display(df.limit(5))


###Criação das Delta Tables

In [0]:
for name, df in dfs.items():
    name = name.replace("df_","")
    df.write.format("delta").mode("overwrite").saveAsTable(name)
    

###Documentação das tabelas

In [0]:
# Comentando todas as tabelas do schema bronze
# As descrições seguem o contexto do dataset Brazilian E-Commerce (Olist)

table_comments = {
    "customers": "Tabela de clientes do e-commerce. Contém informações de localização (CEP, cidade, estado) e identificador único do cliente. Origem: dataset público Olist.",
    "geolocation": "Tabela de geolocalização. Mapeia CEPs brasileiros para latitude, longitude, cidade e estado. Usada para enriquecer dados de localização de clientes e vendedores.",
    "order_items": "Tabela de itens por pedido. Cada linha representa um produto dentro de um pedido, com preço, frete, data limite de entrega e vendedor responsável.",
    "order_payments": "Tabela de pagamentos por pedido. Contém sequência de pagamentos, tipo (cartão, boleto, voucher), parcelas e valor pago.",
    "order_reviews": "Tabela de avaliações de pedidos. Inclui nota (1-5), título, comentário e datas de criação/atualização da avaliação.",
    "orders": "Tabela principal de pedidos. Contém status, datas de compra, aprovação, envio e entrega, além do identificador do cliente.",
    "products": "Tabela de produtos. Contém categoria, dimensões físicas (peso, comprimento, largura, altura) e nome da categoria em português.",
    "sellers": "Tabela de vendedores. Contém identificador do vendedor e localização (CEP, cidade, estado).",
    "product_category_name_translation": "Tabela de tradução de categorias. Mapeia nomes de categorias do português para o inglês."
}

for table_name, comment in table_comments.items():
    spark.sql(f"""
        COMMENT ON TABLE {table_name} IS '{comment}'
    """)
    print(f"Comentário adicionado à tabela {table_name}")

print("\nDocumentação das tabelas concluída!")


####Documentação das colunas

In [0]:
# Comentando as colunas de todas as tabelas do schema bronze[cite: 1]
# As descrições incluem o "Domínio de valores", cumprindo o critério da Etapa 4.3 do MVP.

column_comments = {
    "customers": {
        "customer_id": "Identificador único do cliente (chave primária). Domínio: String (UUID).",
        "customer_unique_id": "Identificador único do cliente real. Domínio: String (UUID).",
        "customer_zip_code_prefix": "Primeiros 5 dígitos do CEP do cliente. Domínio: String numérica (00000 a 99999).",
        "customer_city": "Cidade do cliente. Domínio: Texto livre.",
        "customer_state": "Estado (UF) do cliente. Domínio: Categórico (Siglas de 2 letras dos estados, ex: SP, RJ).",
    },
    "geolocation": {
        "geolocation_zip_code_prefix": "Primeiros 5 dígitos do CEP. Domínio: String numérica (00000 a 99999).",
        "geolocation_lat": "Latitude da localização. Domínio: Numérico decimal (aprox. -33.7 a 5.2).",
        "geolocation_lng": "Longitude da localização. Domínio: Numérico decimal (aprox. -73.9 a -34.8).",
        "geolocation_city": "Cidade correspondente ao CEP. Domínio: Texto livre.",
        "geolocation_state": "Estado (UF) correspondente ao CEP. Domínio: Categórico (Siglas de 2 letras).",
    },
    "order_items": {
        "order_id": "Identificador único do pedido. Domínio: String (UUID).",
        "order_item_id": "Número sequencial que identifica cada item dentro de um pedido. Domínio: Numérico inteiro (1 a 21).",
        "product_id": "Identificador único do produto. Domínio: String (UUID).",
        "seller_id": "Identificador único do vendedor. Domínio: String (UUID).",
        "shipping_limit_date": "Data limite para o envio. Domínio: Data/Hora (Timestamp).",
        "price": "Preço do item. Domínio: Numérico decimal (> 0).",
        "freight_value": "Valor do frete do item. Domínio: Numérico decimal (>= 0).",
    },
    "order_payments": {
        "order_id": "Identificador único do pedido. Domínio: String (UUID).",
        "payment_sequential": "Sequência do pagamento. Domínio: Numérico inteiro (1 a 29).",
        "payment_type": "Tipo de pagamento. Domínio: Categórico ('credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined').",
        "payment_installments": "Número de parcelas. Domínio: Numérico inteiro (0 a 24).",
        "payment_value": "Valor total do pagamento. Domínio: Numérico decimal (>= 0).",
    },
    "order_reviews": {
        "review_id": "Identificador único da avaliação. Domínio: String (UUID).",
        "order_id": "Identificador único do pedido avaliado. Domínio: String (UUID).",
        "review_score": "Nota da avaliação. Domínio: Numérico inteiro (1 a 5).",
        "review_comment_title": "Título do comentário. Domínio: Texto livre ou Nulo.",
        "review_comment_message": "Mensagem do comentário. Domínio: Texto livre ou Nulo.",
        "review_creation_date": "Data de criação da avaliação. Domínio: Data/Hora (Timestamp).",
        "review_answer_timestamp": "Data de resposta do cliente. Domínio: Data/Hora (Timestamp).",
    },
    "orders": {
        "order_id": "Identificador único do pedido (chave primária). Domínio: String (UUID).",
        "customer_id": "Identificador único do cliente. Domínio: String (UUID).",
        "order_status": "Status do pedido. Domínio: Categórico ('created', 'approved', 'invoiced', 'shipped', 'delivered', 'canceled', 'processing', 'unavailable').",
        "order_purchase_timestamp": "Data da compra. Domínio: Data/Hora (Timestamp).",
        "order_approved_at": "Data de aprovação. Domínio: Data/Hora (Timestamp) ou Nulo.",
        "order_delivered_carrier_date": "Data de entrega à transportadora. Domínio: Data/Hora (Timestamp) ou Nulo.",
        "order_delivered_customer_date": "Data de entrega ao cliente. Domínio: Data/Hora (Timestamp) ou Nulo.",
        "order_estimated_delivery_date": "Data estimada de entrega. Domínio: Data/Hora (Timestamp).",
    },
    "products": {
        "product_id": "Identificador único do produto (chave primária). Domínio: String (UUID).",
        "product_category_name": "Nome da categoria. Domínio: Categórico (ex: 'beleza_saude') ou Nulo.",
        "product_name_lenght": "Comprimento do nome do produto. Domínio: Numérico inteiro (5 a 76) ou Nulo.",
        "product_description_lenght": "Comprimento da descrição. Domínio: Numérico inteiro (4 a 3992) ou Nulo.",
        "product_photos_qty": "Quantidade de fotos. Domínio: Numérico inteiro (1 a 20) ou Nulo.",
        "product_weight_g": "Peso do produto em gramas. Domínio: Numérico inteiro (0 a 40425) ou Nulo.",
        "product_length_cm": "Comprimento do produto. Domínio: Numérico inteiro (7 a 105) ou Nulo.",
        "product_height_cm": "Altura do produto. Domínio: Numérico inteiro (2 a 105) ou Nulo.",
        "product_width_cm": "Largura do produto. Domínio: Numérico inteiro (6 a 118) ou Nulo.",
    },
    "sellers": {
        "seller_id": "Identificador único do vendedor. Domínio: String (UUID).",
        "seller_zip_code_prefix": "Primeiros 5 dígitos do CEP do vendedor. Domínio: String numérica (00000 a 99999).",
        "seller_city": "Cidade do vendedor. Domínio: Texto livre.",
        "seller_state": "Estado (UF) do vendedor. Domínio: Categórico (Siglas de 2 letras).",
    },
    "product_category_name_translation": {
        "product_category_name": "Categoria em português. Domínio: Categórico.",
        "product_category_name_english": "Categoria em inglês. Domínio: Categórico.",
    },
}

for table_name, columns in column_comments.items():
    for col_name, comment in columns.items():
        # Escapa aspas simples dentro do comentário[cite: 1]
        safe_comment = comment.replace("'", "''")
        spark.sql(f"""
            ALTER TABLE {table_name} ALTER COLUMN {col_name} COMMENT '{safe_comment}'
        """)
    print(f"Comentários de colunas (com domínio) adicionados à tabela {table_name}")

print("\nDocumentação do Catálogo concluída!")